# Análisis de Eficiencia Espacial: Detour Factor (Factor de Desviación)

El **Detour Factor** (o coeficiente de circuidad) mide la eficiencia de una red de transporte comparando la distancia real recorrida a través de la infraestructura contra la distancia teórica en línea recta (Haversine).

**Fórmula:**
$$DF = \frac{Distancia_{Red} + Distancia_{Caminata}}{Distancia_{Haversine}}$$

Un valor de **1.0** indica una ruta perfectamente directa. Valores superiores a **1.5** suelen indicar ineficiencias topológicas, barreras urbanas o rodeos excesivos por transbordos.

En este notebook realizaremos:
1. Análisis estadístico de 500 viajes aleatorios.
2. Visualización de la trazabilidad y sistemas involucrados.
3. Mapeo interactivo de rutas específicas.

In [1]:
# ==========================================
# 2. Inicialización del Entorno y Librerías
# ==========================================
%load_ext autoreload
%autoreload 2
%matplotlib inline 

import sys
import os
import folium
import warnings
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display


# Silenciar advertencias futuras para un reporte limpio
warnings.filterwarnings('ignore', category=FutureWarning)

# Asegurar que Jupyter encuentra la carpeta raíz del proyecto
proyecto_path = os.path.abspath('..')
if proyecto_path not in sys.path:
    sys.path.append(proyecto_path)
### Visualizador de utils
from src.core.utils.utils_detaur_factor import render_vft_detour_map
from src.infrastructure.go_client.client import fetch_full_network
from src.api.schemas.schemas import GeoJSONTransportSchema
from src.core.services.graph_builder import VFTGraphBuilder
from src.core.algorithms.topologicalIndicators.detaurFactor import DetourFactorOrchestrator
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.family'] = 'sans-serif'

In [2]:
# ==========================================
# EXPORTACIÓN DE ASSETS — VFT Model
# ==========================================
import os
from pathlib import Path

_NB_DIR = Path(os.path.abspath('.'))
IMG_DIR = _NB_DIR / 'ASSETS' / 'img'
HTML_DIR = _NB_DIR / 'ASSETS' / 'img' / 'html'
IMG_DIR.mkdir(parents=True, exist_ok=True)
HTML_DIR.mkdir(parents=True, exist_ok=True)

# URL base GitHub Pages (rama gh-pages, construida por JupyterBook desde main)
GH_PAGES_BASE = "https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html"

def render_table_as_png(df, filename, title="", dpi=150):
    """Renderiza un DataFrame como imagen PNG con estilo VFT."""
    df_str = df.astype(str)
    n_rows, n_cols = df_str.shape
    figsize = (max(8, n_cols * 2.0), max(1.5, n_rows * 0.5 + 1.0))
    fig, ax = plt.subplots(figsize=figsize)
    ax.axis('off')
    if title:
        fig.suptitle(title, fontsize=11, fontweight='bold', y=0.98)
    tbl = ax.table(
        cellText=df_str.values,
        colLabels=df_str.columns.tolist(),
        loc='center',
        cellLoc='center'
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width(col=list(range(n_cols)))
    for j in range(n_cols):
        tbl[(0, j)].set_facecolor('#2C3E50')
        tbl[(0, j)].set_text_props(color='white', fontweight='bold')
    for i in range(1, n_rows + 1):
        for j in range(n_cols):
            tbl[(i, j)].set_facecolor('#EAF2F8' if i % 2 == 0 else 'white')
    path = IMG_DIR / filename
    fig.savefig(path, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print(f"✅ Tabla exportada: {path.name}")
    return path

async def folium_to_png(html_path, png_filename, width=1280, height=800):
    """Convierte un HTML de Folium a PNG usando Playwright async (compatible con Jupyter)."""
    from playwright.async_api import async_playwright
    html_path = Path(html_path)
    png_path = IMG_DIR / png_filename
    async with async_playwright() as p:
        browser = await p.chromium.launch()
        page = await browser.new_page(viewport={'width': width, 'height': height})
        await page.goto(f'file://{html_path.resolve()}')
        await page.wait_for_load_state('networkidle', timeout=15000)
        await page.screenshot(path=str(png_path))
        await browser.close()
    print(f"✅ Mapa PNG exportado: {png_filename}")
    print(f"   QR LaTeX → \\qrcode{{{GH_PAGES_BASE}/{html_path.name}}}")
    return png_path

print("✅ Utilidades de exportación cargadas.")
print(f"   IMG_DIR  → {IMG_DIR}")
print(f"   HTML_DIR → {HTML_DIR}")

✅ Utilidades de exportación cargadas.
   IMG_DIR  → /home/galigaribaldi/Documentos/Code/VFTModel/notebooks/ASSETS/img
   HTML_DIR → /home/galigaribaldi/Documentos/Code/VFTModel/notebooks/ASSETS/img/html


In [3]:
# ==========================================
# 3. Descarga de Red y Construcción del Grafo
# ==========================================
print("📥 Descargando red de transporte...")
raw_data = await fetch_full_network()

print("⚙️ Construyendo Grafo Dirigido VFT...")
validated_payload = GeoJSONTransportSchema(**raw_data)

plt.ioff() 
GRAF= VFTGraphBuilder(validated_payload)
G = GRAF.build_graph()
plt.close('all')
plt.ion()

orchestrator = DetourFactorOrchestrator(G)
print(f"✅ Grafo construido: {G.number_of_nodes()} nodos topológicos.")

📥 Descargando red de transporte...
2026-05-16 13:11:21 | INFO     | VFT_Model | Construyendo el puente hacia el módulo espacial de Go...
2026-05-16 13:11:21 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonEstacion
2026-05-16 13:11:21 | INFO     | VFT_Model | Solicitando capa espacial a: http://localhost:8080/movilidad/mapas/geojsonLinea
2026-05-16 13:11:22 | INFO     | VFT_Model | Red extraída: 22766 entidades espaciales listas para VFT.
⚙️ Construyendo Grafo Dirigido VFT...
2026-05-16 13:11:22 | INFO     | VFT_Model | Iniciando VFTGraphBuilder en modo: REALISTIC_INTEGRATION
2026-05-16 13:11:22 | INFO     | VFT_Model | Fase 1 y 2: Extrayendo nodos y trazos base...
2026-05-16 13:11:24 | INFO     | VFT_Model | Fase 2 completada: 64459 aristas interestación creadas.
2026-05-16 13:11:24 | INFO     | VFT_Model | Grafo Base construido: 11115 Nodos y 32103 Segmentos.
2026-05-16 13:11:24 | WARNING  | VFT_Model | Eliminadas 1543 aristas phantom

## 1. Muestreo Estadístico y Caso de Prueba Inicial
En esta sección generamos una muestra aleatoria de 100 viajes para entender el comportamiento base de la red y visualizamos el primer resultado obtenido sin ningún criterio de ordenamiento.

In [4]:
# 1. Generar la muestra estadística global
tamano_muestra = 100
print(f"🎲 Generando muestra de {tamano_muestra} rutas...")
muestra_json = orchestrator.calculate_sample_routes(sample_size=tamano_muestra, seed=42, return_json=True)

if muestra_json:
    # Crear DataFrame de métricas
    df_metrics = pd.DataFrame([m['metrics'] for m in muestra_json])
    
    # Resumen estadístico
    print("\n📊 ESTADÍSTICAS GLOBALES DEL DETOUR FACTOR:")
    _df_desc = df_metrics['Factor_Desviacion'].describe().to_frame().T
    display(_df_desc)
    render_table_as_png(
        _df_desc.reset_index(drop=True),
        'NB04_tab01_estadisticas_globales_df.png',
        'Estadísticas Globales — Detour Factor (n=100)'
    )
    
    # Visualizar el primer caso de la lista
    primer_caso = muestra_json[0]
    mapa_ini, _, _, _, _, _ = render_vft_detour_map(
        primer_caso, 
        title=f"CASO INICIAL ALEATORIO (DF: {primer_caso['metrics']['Factor_Desviacion']})"
    )
    _html = HTML_DIR / 'NB04_mapa01_caso_inicial.html'
    mapa_ini.save(str(_html))
    await folium_to_png(_html, 'NB04_mapa01_caso_inicial.png')
    display(mapa_ini)

🎲 Generando muestra de 100 rutas...

📊 ESTADÍSTICAS GLOBALES DEL DETOUR FACTOR:


,count,mean,std,min,25%,50%,75%,max
Factor_Desviacion,100.0,1.5455,0.51704,0.97,1.27,1.4,1.67,4.34


✅ Tabla exportada: NB04_tab01_estadisticas_globales_df.png
🗺️ CASO INICIAL ALEATORIO (DF: 1.04)
📍 Punto cerca de Tepehuanos La Joya ➡️ Punto cerca de Acoxpa
📊 DF: 1.04 | Red: 4.04km vs Teórica: 3.87km
✅ Mapa PNG exportado: NB04_mapa01_caso_inicial.png
   QR LaTeX → \qrcode{https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html/NB04_mapa01_caso_inicial.html}


## 2. Análisis de Máxima Eficiencia: Las 5 Mejores Rutas
Aquí identificamos los trayectos donde la red de transporte es más competitiva frente al automóvil (trayectos casi directos). Visualizamos el mapa de la ruta con el menor Factor de Desviación.

In [5]:
# 1. Mostrar tabla de las 5 mejores
print("✅ TOP 5 RUTAS MÁS EFICIENTES (Cercanas a 1.0):")
df_mejores = df_metrics.sort_values(by="Factor_Desviacion", ascending=True).head(5)
_cols_rutas = ['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']
display(df_mejores[_cols_rutas])
render_table_as_png(
    df_mejores[_cols_rutas].reset_index(drop=True),
    'NB04_tab02_top5_rutas_eficientes.png',
    'Top 5 Rutas Más Eficientes — Detour Factor'
)

# 2. Identificar y graficar el mejor caso absoluto
mejor_caso = min(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_mejor, _, _, _, _, _ = render_vft_detour_map(
    mejor_caso, 
    title=f"LA RUTA MÁS EFICIENTE (DF: {mejor_caso['metrics']['Factor_Desviacion']})"
)
_html = HTML_DIR / 'NB04_mapa02_ruta_mas_eficiente.html'
mapa_mejor.save(str(_html))
await folium_to_png(_html, 'NB04_mapa02_ruta_mas_eficiente.png')
display(mapa_mejor)

✅ TOP 5 RUTAS MÁS EFICIENTES (Cercanas a 1.0):


,Origen,Destino,Sistemas_Involucrados,Distancia_Red_km,Factor_Desviacion
82,Punto cerca de Plutarco Elias Calles,Punto cerca de Fuente de Loreto - Av. Guelatao,[RTP],0.99,0.97
85,Punto cerca de Norte 84,Punto cerca de Misterios,"[TROLE, RTP]",2.53,1.00
13,Punto cerca de C. C. Arenal Cuarta Sección - X...,Punto cerca de Iztacihuatl - Cacamatzin,[RTP],0.72,1.01
0,Punto cerca de Tepehuanos La Joya,Punto cerca de Acoxpa,[RTP],4.04,1.04
79,Punto cerca de Paseo de la Reforma - Fuente de...,Punto cerca de El Ángel,"[CC, MB, TROLE]",3.61,1.05


✅ Tabla exportada: NB04_tab02_top5_rutas_eficientes.png
🗺️ LA RUTA MÁS EFICIENTE (DF: 0.97)
📍 Punto cerca de Plutarco Elias Calles ➡️ Punto cerca de Fuente de Loreto - Av. Guelatao
📊 DF: 0.97 | Red: 0.99km vs Teórica: 1.02km
✅ Mapa PNG exportado: NB04_mapa02_ruta_mas_eficiente.png
   QR LaTeX → \qrcode{https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html/NB04_mapa02_ruta_mas_eficiente.html}


## 3. Análisis de Puntos Críticos: Las 5 Peores Rutas
Identificamos los trayectos que presentan mayores ineficiencias, ya sea por falta de conexiones directas o rodeos excesivos. Mapeamos el "Peor Caso" para analizar visualmente la falla en la topología.

In [6]:
# 1. Mostrar tabla de las 5 peores
print("🚨 TOP 5 RUTAS CON MAYOR INEFICIENCIA (Rodeos Críticos):")
df_peores = df_metrics.sort_values(by="Factor_Desviacion", ascending=False).head(10)
_cols_rutas = ['Origen', 'Destino', 'Sistemas_Involucrados', 'Distancia_Red_km', 'Factor_Desviacion']
display(df_peores[_cols_rutas])
render_table_as_png(
    df_peores[_cols_rutas].reset_index(drop=True),
    'NB04_tab03_top10_rutas_ineficientes.png',
    'Top 10 Rutas Más Ineficientes — Detour Factor'
)

# 2. Identificar y graficar el peor caso absoluto
peor_caso = max(muestra_json, key=lambda x: x['metrics']['Factor_Desviacion'])
mapa_peor, _, _, _, _, _ = render_vft_detour_map(
    peor_caso, 
    title=f"LA RUTA MÁS INEFICIENTE (DF: {peor_caso['metrics']['Factor_Desviacion']})"
)
_html = HTML_DIR / 'NB04_mapa03_ruta_mas_ineficiente.html'
mapa_peor.save(str(_html))
await folium_to_png(_html, 'NB04_mapa03_ruta_mas_ineficiente.png')
display(mapa_peor)

🚨 TOP 5 RUTAS CON MAYOR INEFICIENCIA (Rodeos Críticos):


,Origen,Destino,Sistemas_Involucrados,Distancia_Red_km,Factor_Desviacion
37,Punto cerca de Carlos A. Vidal - Everardo Gonz...,Punto cerca de 20 de Noviembre - Gargolas,"[RTP, METRO, TL, CC]",49.74,4.34
33,Punto cerca de Guelatao,Punto cerca de Base Luna y Plutón,"[METRO, TROLE, CC]",33.55,3.85
57,Punto cerca de Las Fuentes,Punto cerca de Cuauhtémoc,[MEXIBÚS],1.29,3.51
99,Punto cerca de Industria Militar - Gral. Ildef...,Punto cerca de Cerrada de las Flores,"[CC, METRO, RTP]",27.11,2.52
86,Punto cerca de Av. Sur del Comercio,Punto cerca de Calz. de Tlalpan - División del...,"[RTP, METRO, TROLE, CC, TL]",41.19,2.34
71,Punto cerca de Terminal Aérea,Punto cerca de Pekin y Japón,"[TROLE, METRO, CC]",2.37,2.24
22,Punto cerca de San Juan de Aragón,Punto cerca de Circuito Interior - Av. Benjamí...,"[MB, TROLE, METRO, RTP]",25.72,2.19
36,Punto cerca de Gobernador Carlos Hank González...,Punto cerca de Parotas - Sinaloa,"[CC, RTP, METRO, MB]",38.73,2.12
7,Punto cerca de Andrés Molina y Playa Icacos,Punto cerca de Marina Nacional - Lago Bolsena,"[CC, METRO]",17.41,2.09
48,Punto cerca de Av. Chapultepec - Napoles,Punto cerca de Marina Nacional - Carrillo Puerto,"[RTP, METRO, CC]",8.68,2.09


✅ Tabla exportada: NB04_tab03_top10_rutas_ineficientes.png
🗺️ LA RUTA MÁS INEFICIENTE (DF: 4.34)
📍 Punto cerca de Carlos A. Vidal - Everardo González ➡️ Punto cerca de 20 de Noviembre - Gargolas
📊 DF: 4.34 | Red: 49.74km vs Teórica: 11.46km
✅ Mapa PNG exportado: NB04_mapa03_ruta_mas_ineficiente.png
   QR LaTeX → \qrcode{https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html/NB04_mapa03_ruta_mas_ineficiente.html}


## 4. Análisis de Ruta Arbitraria Personalizada (Caso de Estudio)
En esta sección evaluamos un viaje específico utilizando coordenadas exactas (Longitud, Latitud). Esto permite analizar la "primera y última milla" (caminata hacia la estación más cercana) y el factor de desviación para trayectos reales.

**Caso de Estudio:**
* **Origen:** Zona Sur, fuera del Anillo Periférico.
* **Destino:** Zona Oriente, inmediaciones del Autódromo Hermanos Rodríguez.

## 5. Galería de Casos de Estudio: Análisis de Trayectos Específicos
En esta sección, evaluamos una serie de rutas estratégicas seleccionadas por su relevancia en la conectividad de la Ciudad de México. 

Cada caso analiza:
1. **Contexto Geográfico:** Por qué son importantes estos puntos.
2. **Eficiencia Topológica:** Cómo se comporta la red frente a la distancia directa.
3. **Métrica de Caminata:** La distancia necesaria para conectar con la red formal (Primera y Última Milla).

In [7]:
from IPython.display import display, Markdown

# 1. DEFINICIÓN DE RUTAS (Agrega aquí tantas como necesites)
casos_estudio = [
    {
        "titulo": "Caso 1: Periferia Extrema Sur hacia Centro-Oriente",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico hacia el Hub de entretenimiento del Autódromo. Evalúa la dificultad de salir de zonas con baja densidad de transporte formal.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.09342206113304, 19.4059272938029)
    },
    {
        "titulo": "Caso 2: Conectividad Sur-Centro/Poniente (Milpa Alta a Reforma)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Reforma.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.17161535569772, 19.42527244485531) 
    },
    {
        "titulo": "Caso 3: Conectividad Sur-Poniente (Milpa Alta a Santa Fé)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el CBD Santa Fé.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.26937963016258 ,19.357049109027344) 
    },
    {
        "titulo": "Caso 4: Conectividad Sur-Poniente Norte (Milpa Alta al punto más cercano a FES Acatlán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 5: Conectividad Sur-Centro/Sur (Milpa Alta a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia el Centro Educativo Ciudad Universitaria.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.184797668633, 19.33163949142085)
    },
    {
        "titulo": "Caso 6: Conectividad Sur-Centro/Sur (Milpa Alta a Coyoacán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia Zona Residencial Centro Sur coyoacán.",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.15633349395276, 19.342445017191288)
    },
    {
        "titulo": "Caso 7: Conectividad Oriente-Centro/Sur (Santa Catarina a CU)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo Ciudad Universitaria.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.184797668633, 19.33163949142085) 
    },
    {
        "titulo": "Caso 8: Conectividad Oriente-Centro/Sur (Santa Catarina al Punto más cercano a FES Acatán)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Oriente) hacia Centro Educativo FES Acatlán.",
        "coords_o": (-98.96533517329291, 19.315314989332904),
        "coords_d": (-99.2098132566566, 19.489900115538866) 
    },
    {
        "titulo": "Caso 9: Conectividad Sur-Oriente (Milpa Alta - Tláhuac/Santa Catarina)",
        "descripcion": "Trayecto desde zonas residenciales fuera del Anillo Periférico (Sur) hacia zonas residenciales fuera del Anillo Periférico (Oriente).",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-98.9648158208182, 19.315197623324714) 
    },
    {
        "titulo": "Caso 10",
        "descripcion": "Milpa Alta - Ajusco",
        "coords_o": (-99.04747820950537, 19.20745080334388),
        "coords_d": (-99.22739372393065, 19.280519282861828) 
    },
    {
        "titulo": "Caso 11",
        "descripcion": "Ajusco - Prepa 1",
        "coords_o": (-99.22739372393065, 19.280519282861828),
        "coords_d": (-99.12138370600817, 19.27230448210211) 
    },
    {
        "titulo": "Caso 11",
        "descripcion": "Parque de Los olivos - Centro de Tláhuac",
        "coords_o": (-99.00381778682763, 19.251950791794982),
        "coords_d": (-99.00476113482334, 19.27063980641322) 
    }
]

# 2. BUCLE OPTIMIZADO DE RENDERIZADO
# Mapas 01-03 ya usados arriba; casos de estudio → mapa04 en adelante
for idx, caso in enumerate(casos_estudio, 1):
    # Presentación elegante del título y descripción
    display(Markdown(f"### {caso['titulo']}"))
    display(Markdown(f"**Contexto:** {caso['descripcion']}"))
    
    # Cálculo
    res = orchestrator.calculate_custom_route(caso['coords_o'], caso['coords_d'], return_json=True)
    
    if res:
        # Renderizado del mapa y obtención de métricas robustas
        mapa, ini, fin, df_val, d_hav, d_red = render_vft_detour_map(
            res, 
            title=f"Visualización: {caso['titulo']}"
        )
        
        # Resumen de métricas en formato Markdown para mejor lectura
        metricas_caminata = res['metrics'].get('Consideraciones_Reales', {})
        c1 = metricas_caminata.get('Caminata_1ra_Milla_Metros', 0)
        c2 = metricas_caminata.get('Caminata_Ultima_Milla_Metros', 0)
        
        resumen_md = f"""
| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | {metricas_caminata.get('Estacion_Abordaje', 'N/A')} |
| **Punto de Descenso** | {metricas_caminata.get('Estacion_Descenso', 'N/A')} |
| **Factor de Desviación** | `{df_val}` |
| **Distancia en Red** | {d_red} km |
| **Tiempo Promedio en reocorrer la Red(20 km/hr)** | {float(d_red)/20} km |
| **Tiempo Promedio en reocorrer la Distancia Geográfica(20 km/hr)** | {float(d_hav)/20}hr,min |
| **Distancia en Haversine** | {d_hav} km |
| **Caminata Total (1ra + Última)** | {round(c1 + c2, 2)} metros |
"""
        display(Markdown(resumen_md))

        # Exportar mapa del caso
        _mapa_num = idx + 3  # mapa01-03 ya usados
        _html = HTML_DIR / f'NB04_mapa{_mapa_num:02d}_caso{idx:02d}.html'
        mapa.save(str(_html))
        await folium_to_png(_html, f'NB04_mapa{_mapa_num:02d}_caso{idx:02d}.png')

        display(mapa)
        display(Markdown("---")) # Separador visual entre mapas
    else:
        display(Markdown(f"⚠️ *No se pudo calcular la ruta para: {caso['titulo']}*"))

### Caso 1: Periferia Extrema Sur hacia Centro-Oriente

**Contexto:** Trayecto desde zonas residenciales fuera del Anillo Periférico hacia el Hub de entretenimiento del Autódromo. Evalúa la dificultad de salir de zonas con baja densidad de transporte formal.

🗺️ Visualización: Caso 1: Periferia Extrema Sur hacia Centro-Oriente
📍 Punto cerca de Niños Héroes y San Gregorio - Oaxtepec ➡️ Punto cerca de Viaducto y Puerta 5
📊 DF: 1.29 | Red: 29.18km vs Teórica: 22.59km



| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | Niños Héroes y San Gregorio - Oaxtepec |
| **Punto de Descenso** | Viaducto y Puerta 5 |
| **Factor de Desviación** | `1.29` |
| **Distancia en Red** | 29.18 km |
| **Tiempo Promedio en reocorrer la Red(20 km/hr)** | 1.459 km |
| **Tiempo Promedio en reocorrer la Distancia Geográfica(20 km/hr)** | 1.1295hr,min |
| **Distancia en Haversine** | 22.59 km |
| **Caminata Total (1ra + Última)** | 0 metros |


✅ Mapa PNG exportado: NB04_mapa04_caso01.png
   QR LaTeX → \qrcode{https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html/NB04_mapa04_caso01.html}


...

**Contexto:** Parque de Los olivos - Centro de Tláhuac

🗺️ Visualización: Caso 11
📍 Punto cerca de Isidro Tapia ➡️ Punto cerca de Ferrocarril San Rafael Atlixco - Miguel Hidalgo
📊 DF: 1.06 | Red: 2.2km vs Teórica: 2.08km



| Métrica | Valor |
| :--- | :--- |
| **Punto de Abordaje** | Isidro Tapia |
| **Punto de Descenso** | Ferrocarril San Rafael Atlixco - Miguel Hidalgo |
| **Factor de Desviación** | `1.06` |
| **Distancia en Red** | 2.2 km |
| **Tiempo Promedio en reocorrer la Red(20 km/hr)** | 0.11000000000000001 km |
| **Tiempo Promedio en reocorrer la Distancia Geográfica(20 km/hr)** | 0.10400000000000001hr,min |
| **Distancia en Haversine** | 2.08 km |
| **Caminata Total (1ra + Última)** | 0 metros |


✅ Mapa PNG exportado: NB04_mapa15_caso12.png
   QR LaTeX → \qrcode{https://galigaribaldi.github.io/VFTModel/notebooks/ASSETS/img/html/NB04_mapa15_caso12.html}


---